## Imports and data loading

In [1]:
import math
import random
import time

import pandas as pd
import numpy as np

from scipy import sparse
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity

from sparselsh import LSH

## Content-based, regression

In [2]:
# load food.com data
directory = 'data/food.com'
df_recipe_rating = pd.read_csv(f'{directory}/recipe_ratings.csv')
df_recipe = pd.read_csv(f'{directory}/recipe.csv')

In [3]:
df_recipe.head()

,recipe_id,name,minutes,n_steps,n_ingredients,calories,fat,sugar,sodium,protein,saturated,carbs
0,137739,arriba baked winter squash mexican style,55,11,7,51.5,0.0,13.0,0.0,2.0,0.0,4.0
1,31490,a bit different breakfast pizza,30,9,6,173.4,18.0,0.0,17.0,22.0,35.0,1.0
2,112140,all in the kitchen chili,130,6,13,269.8,22.0,32.0,48.0,39.0,27.0,5.0
3,59389,alouette potatoes,45,11,11,368.1,17.0,10.0,2.0,14.0,8.0,20.0
4,44061,amish tomato ketchup for canning,190,5,8,352.9,1.0,337.0,23.0,3.0,0.0,28.0


In [4]:
df_recipe_rating

,user_id,recipe_id,rating
0,2046,4684,5.0
1,2046,517,5.0
2,1773,7435,5.0
3,1773,278,4.0
4,2046,3431,5.0
...,...,...,...
698896,926904,457971,5.0
698897,2002312797,27208,5.0
698898,1290903,131607,5.0
698899,226867,363072,5.0


In [60]:
#EXERCISE: Build a content-based recommender system that uses linear regression 
#          to predict ratings.
#          Try it out on users with a high number of ratings.   
#          Try some train-test split to evaluate performance.

# find users with many ratings AND high variance
user_stats = df_recipe_rating.groupby('user_id')['rating'].agg(['count', 'std'])
top_users = user_stats[(user_stats['count'] > 100) & (user_stats['std'] > 1.0)]
top_users = top_users.sort_values('std', ascending=False)
print(top_users.head(10))

# pick a user with a high number of ratings
target_id = top_users.index[5]
print(f"Selected user: {target_id}")

# select the ratings of a specific user 
df_user = df_recipe_rating[df_recipe_rating['user_id']==target_id][['recipe_id','rating']] # recipes for which the user `target_id` has given ratings
df_rec = pd.merge(df_user, df_recipe, on='recipe_id', how='inner') # merge with the recipe data to get the features of the recipes  

df_rec.head()

# df_user['rating'].value_counts()
# df_user['rating'].describe()


         count       std
user_id                 
202661     111  2.283079
220195     127  2.069261
904483     120  2.025036
841895     152  2.021570
502302     192  1.953624
184530     205  1.949236
583193     135  1.935471
109110     106  1.915112
207176     462  1.900351
201584     119  1.860924
Selected user: 184530


,recipe_id,rating,name,minutes,n_steps,n_ingredients,calories,fat,sugar,sodium,protein,saturated,carbs
0,65356,5.0,roasted jumbo shrimp with potatoes lemon and ...,60,12,9,305.6,18.0,4.0,69.0,50.0,8.0,7.0
1,100081,4.0,absolutely delicious gourmet prawns,30,7,7,214.3,21.0,3.0,29.0,32.0,23.0,1.0
2,10513,4.0,peppered prawns,15,9,9,184.1,19.0,3.0,22.0,21.0,37.0,2.0
3,125457,4.0,spinach casserole au gratin,75,5,10,170.3,8.0,17.0,12.0,27.0,11.0,6.0
4,74680,4.0,shrimp and asparagus in dill sauce,30,9,11,638.1,24.0,15.0,28.0,64.0,18.0,30.0


In [61]:
# define features
# features = ['minutes', 'n_steps', 'n_ingredients', 'calories', 'fat', 'sugar', 'sodium', 'protein', 'saturated', 'carbs']
features = ['calories', 'fat', 'sugar', 'protein']

# split training and test
# 80/20 split
train_size = int(0.8 * len(df_rec))
train_df = df_rec[:train_size]
test_df = df_rec[train_size:]

# fit on the training, test on the rest
model = LinearRegression()
model.fit(train_df[features], train_df['rating'])

predictions = model.predict(test_df[features])

# evaluate performance
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(test_df['rating'], predictions)

print(f'Mean Squared Error: {mse}')

baseline_prediction = train_df['rating'].mean()
baseline_mse = mean_squared_error(test_df['rating'], [baseline_prediction] * len(test_df))
print(f'Baseline Mean Squared Error: {baseline_mse}')

Mean Squared Error: 7.565629202922329
Baseline Mean Squared Error: 7.723713563355146


### Reflections

- seems like recommending food based on nutritional values does not generally work (maybe except for bodybuilders)

## Content-based, KNN (with LSH)

In [68]:
directory = 'data/movielens/ml-latest-small'
#directory = 'data/movielens/ml-latest' #change into this one for the full dataset (slow)

df_movies = pd.read_csv(f'{directory}/movies.csv')
df_ratings = pd.read_csv(f'{directory}/ratings.csv')
df_tags = pd.read_csv(f'{directory}/tags.csv')

#transform tags such that they are lower-case, single-word tokens
df_tags['tag'] = df_tags['tag'].apply(lambda x: str(x).lower().replace(' ', '_'))

In [8]:
df_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [9]:
df_tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,highly_quotable,1445714996
2,2,60756,will_ferrell,1445714992
3,2,89774,boxing_story,1445715207
4,2,89774,mma,1445715200


### Step1: Calculate item profiles

In [69]:
# calculates the lexicon of most frequent tags.
tag_frequency_threshold = 5 # increase number to filter
df_lexicon = df_tags.groupby('tag').size().reset_index(name='count') # get a dataframe with tags and respective counts
df_lexicon = df_lexicon[df_lexicon['count'] >= tag_frequency_threshold] # filter out tags that are not frequent enough
print(df_lexicon.head())

# discard movies with no tags
df_tags = df_tags[df_tags['tag'].isin(df_lexicon['tag'])] # filter tags to only include those in the lexicon

# you can drop the userId and timestamp columns because we don't care who assigned the tag and when
df_tags = df_tags.drop(columns=['userId', 'timestamp'])

df_tags.head()

             tag  count
15       aardman      5
23        action     16
26  adam_sandler      5
28   adolescence     11
33      adultery     11


,movieId,tag
0,60756,funny
2,60756,will_ferrell
6,106782,drugs
7,106782,leonardo_dicaprio
10,431,al_pacino


In [70]:
#calculate the sparse feature vector based on the TF-IDF of words in documents
#the TF-IDF vectors are saved as sparse representations into the dataframe
df_features = df_tags.groupby('movieId').agg(lambda x: ' '.join(x)).reset_index()
vectorizer = TfidfVectorizer(tokenizer=lambda x: x.split(' ')).fit(sorted(df_features['tag']))
vectorizer.vocabulary_
df_features['feature_vector'] = df_features['tag'].apply(lambda x : vectorizer.transform([x]))
df_features.head()

/home/rafael/itu/ds-in-prod/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,movieId,tag,feature_vector
0,1,pixar pixar fun,<Compressed Sparse Row sparse matrix of dtype ...
1,2,fantasy,<Compressed Sparse Row sparse matrix of dtype ...
2,5,pregnancy remake,<Compressed Sparse Row sparse matrix of dtype ...
3,7,remake,<Compressed Sparse Row sparse matrix of dtype ...
4,11,politics president,<Compressed Sparse Row sparse matrix of dtype ...


### Step2: Index item profiles into LSH

In [12]:
#index all item vectors into LSH
lsh = LSH(
    hash_size=8,
    input_dim=len(vectorizer.vocabulary_),
    num_hashtables=4
)

for _, row in df_features.iterrows():
    lsh.index(row['feature_vector'], extra_data=row['movieId'])

#run an example query to the LSH
example_vector = df_features['feature_vector'].iloc[0] # example vector to query
lsh.query(example_vector, num_results=5, distance_func='cosine')

[((<Compressed Sparse Row sparse matrix of dtype 'float64'
   	with 2 stored elements and shape (1, 184)>,
   1),
  0.0),
 ((<Compressed Sparse Row sparse matrix of dtype 'float64'
   	with 2 stored elements and shape (1, 184)>,
   1),
  0.0),
 ((<Compressed Sparse Row sparse matrix of dtype 'float64'
   	with 2 stored elements and shape (1, 184)>,
   1),
  0.0),
 ((<Compressed Sparse Row sparse matrix of dtype 'float64'
   	with 2 stored elements and shape (1, 184)>,
   1),
  0.0),
 ((<Compressed Sparse Row sparse matrix of dtype 'float64'
   	with 1 stored elements and shape (1, 184)>,
   2355),
  0.10045313809877454)]

### Step 3: Calculate user profile

In [13]:
# restricts the ratings to the set of most popular movies (optional, not needed for content-based)
numratings_threshold = 0 #increase this number if you want to filter
df_item_popularity = df_ratings[['movieId','rating']].groupby('movieId').count().reset_index()
df_item_popularity.columns = ['movieId','count'] 
df_item_popularity = df_item_popularity.sort_values(by='count', ascending=False)
df_item_popularity = df_item_popularity[df_item_popularity['count'] >= numratings_threshold]
print(f'Number of movies reduced from {len(df_ratings.movieId.unique())} to {len(df_item_popularity.movieId.unique())}')
df_ratings = pd.merge(df_ratings, df_item_popularity, on='movieId', how='inner')[['userId', 'movieId', 'rating']]
df_ratings = df_ratings.sort_values(by='userId')

#rescale the ratings by the user's individual average 
df_ratings['rating_scaled'] = df_ratings.groupby('userId')['rating'].transform(lambda x: x - x.mean())

df_ratings.head()

Number of movies reduced from 9724 to 9724


,userId,movieId,rating,rating_scaled
102,1,1587,5.0,0.633621
94,1,1408,3.0,-1.366379
93,1,1396,3.0,-1.366379
108,1,1732,5.0,0.633621
107,1,1676,3.0,-1.366379


In [14]:
# join ratings with movie feature vectors
df_profile = pd.merge(df_ratings, df_features[['movieId','feature_vector']],
              on='movieId')
#scaling feature vector by rating (this will take a few minutes)
df_profile['feature_vector_scaled'] = df_profile['rating_scaled'] * df_profile['feature_vector']
df_profile

,userId,movieId,rating,rating_scaled,feature_vector,feature_vector_scaled
0,1,1732,5.0,0.633621,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
1,1,1625,5.0,0.633621,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
2,1,1617,5.0,0.633621,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
3,1,1580,3.0,-1.366379,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
4,1,1954,5.0,0.633621,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
...,...,...,...,...,...,...
34528,610,152077,4.0,0.311444,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
34529,610,158872,3.5,-0.188556,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
34530,610,168252,5.0,1.311444,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...
34531,610,168248,5.0,1.311444,<Compressed Sparse Row sparse matrix of dtype ...,<Compressed Sparse Row sparse matrix of dtype ...


In [15]:
start = time.time()
#stack all sparse vectors of user's movies
df_user_vectors = df_profile[['userId', 'feature_vector_scaled']].groupby('userId').agg(sparse.vstack).reset_index()
#compute the average of the vectors without considering the zero entries (this will take a while)
df_user_vectors['feature_vector_scaled'] = df_user_vectors['feature_vector_scaled'].apply(lambda x: csr_matrix(np.nan_to_num(x.sum(axis=0)/x.getnnz(axis=0), 0)))
end = time.time()
print(end - start)
df_user_vectors

/tmp/ipykernel_468880/3630376811.py:5: RuntimeWarning: invalid value encountered in divide
  df_user_vectors['feature_vector_scaled'] = df_user_vectors['feature_vector_scaled'].apply(lambda x: csr_matrix(np.nan_to_num(x.sum(axis=0)/x.getnnz(axis=0), 0)))


0.7071144580841064


,userId,feature_vector_scaled
0,1,<Compressed Sparse Row sparse matrix of dtype ...
1,2,<Compressed Sparse Row sparse matrix of dtype ...
2,3,<Compressed Sparse Row sparse matrix of dtype ...
3,4,<Compressed Sparse Row sparse matrix of dtype ...
4,5,<Compressed Sparse Row sparse matrix of dtype ...
...,...,...
605,606,<Compressed Sparse Row sparse matrix of dtype ...
606,607,<Compressed Sparse Row sparse matrix of dtype ...
607,608,<Compressed Sparse Row sparse matrix of dtype ...
608,609,<Compressed Sparse Row sparse matrix of dtype ...


### Step 4: Rank potential recommendation candidates

In [16]:
#pick a target user to provide recommendations to
idx = 42
target_userId = df_user_vectors.iloc[idx].userId

In [17]:
#get user rating history
df_user_history = (
    df_ratings[df_ratings['userId'] == target_userId]
    .merge(df_movies, on='movieId')
    .sort_values(by='rating', ascending=False)
)

# users profile vector
user_vec = df_user_vectors.loc[
    df_user_vectors['userId'] == target_userId, 'feature_vector_scaled'
].iloc[0]

# query LSH for movies
results = lsh.query(user_vec, num_results=20, distance_func='cosine')

# results is a list of ((vector, extra_data), distance) tuples
candidate_ids = [r[0][1] for r in results]
candidate_dist = [r[1] for r in results]

#select candidate recommendations to user
df_recommendation = (
    pd.DataFrame({'movieId': candidate_ids, 'distance': candidate_dist})
    .merge(df_movies, on='movieId')
    # remove movies the user has already rated
    .loc[lambda d: ~d['movieId'].isin(df_user_history['movieId'])]
    .sort_values('distance')
)

In [18]:
df_recommendation

,movieId,distance,title,genres
5,1028,0.858527,Mary Poppins (1964),Children|Comedy|Fantasy|Musical
6,1025,0.858527,"Sword in the Stone, The (1963)",Animation|Children|Fantasy|Musical
7,1022,0.858527,Cinderella (1950),Animation|Children|Fantasy|Musical|Romance
8,1030,0.858527,Pete's Dragon (1977),Adventure|Animation|Children|Musical
9,1032,0.858527,Alice in Wonderland (1951),Adventure|Animation|Children|Fantasy|Musical
10,1033,0.858527,"Fox and the Hound, The (1981)",Animation|Children|Drama
11,1029,0.858527,Dumbo (1941),Animation|Children|Drama|Musical
12,2080,0.858527,Lady and the Tramp (1955),Animation|Children|Comedy|Romance
13,2078,0.858527,"Jungle Book, The (1967)",Animation|Children|Comedy|Musical
14,2054,0.858527,"Honey, I Shrunk the Kids (1989)",Adventure|Children|Comedy|Fantasy|Sci-Fi


In [19]:
df_user_history.head(10)

,userId,movieId,rating,rating_scaled,title,genres
112,43,810,5.0,0.447368,Kazaam (1996),Children|Comedy|Fantasy
99,43,277,5.0,0.447368,Miracle on 34th Street (1994),Drama
100,43,616,5.0,0.447368,"Aristocats, The (1970)",Animation|Children
101,43,631,5.0,0.447368,All Dogs Go to Heaven 2 (1996),Adventure|Animation|Children|Fantasy|Musical|R...
102,43,648,5.0,0.447368,Mission: Impossible (1996),Action|Adventure|Mystery|Thriller
103,43,661,5.0,0.447368,James and the Giant Peach (1996),Adventure|Animation|Children|Fantasy|Musical
104,43,711,5.0,0.447368,Flipper (1996),Adventure|Children
9,43,1356,5.0,0.447368,Star Trek: First Contact (1996),Action|Adventure|Sci-Fi|Thriller
10,43,1105,5.0,0.447368,Children of the Corn IV: The Gathering (1996),Horror
11,43,1084,5.0,0.447368,Bonnie and Clyde (1967),Crime|Drama


In [20]:
df_user_history.tail(10)

,userId,movieId,rating,rating_scaled,title,genres
49,43,110,3.0,-1.552632,Braveheart (1995),Action|Drama|War
50,43,288,3.0,-1.552632,Natural Born Killers (1994),Action|Crime|Thriller
28,43,236,3.0,-1.552632,French Kiss (1995),Action|Comedy|Romance
29,43,231,3.0,-1.552632,Dumb & Dumber (Dumb and Dumber) (1994),Adventure|Comedy
25,43,539,3.0,-1.552632,Sleepless in Seattle (1993),Comedy|Drama|Romance
18,43,377,3.0,-1.552632,Speed (1994),Action|Romance|Thriller
17,43,380,3.0,-1.552632,True Lies (1994),Action|Adventure|Comedy|Romance|Thriller
16,43,300,3.0,-1.552632,Quiz Show (1994),Drama
98,43,296,3.0,-1.552632,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
113,43,349,3.0,-1.552632,Clear and Present Danger (1994),Action|Crime|Drama|Thriller


### Step 5: Predict ratings of candidate items

In [21]:
#index all user vectors into LSH
df_usr = df_profile[df_profile['userId'] == target_userId]
lsh_usr = LSH(
    hash_size=8,
    input_dim=len(vectorizer.vocabulary_),
    num_hashtables=4
)
for _, row in df_usr.iterrows():
    lsh_usr.index(row['feature_vector'], extra_data=(row['movieId'], row['rating'])) # repeat for all users. Insert movieid and rating as extra data for future retrieval
lsh_usr

candidate_vec = df_features.loc[df_features['movieId'] == candidate_ids[0], 'feature_vector'].iloc[0]                 
lsh_usr.query(candidate_vec, num_results=5, distance_func='cosine')                                                   


[((<Compressed Sparse Row sparse matrix of dtype 'float64'
   	with 1 stored elements and shape (1, 184)>,
   (596, 5.0)),
  0.0),
 ((<Compressed Sparse Row sparse matrix of dtype 'float64'
   	with 1 stored elements and shape (1, 184)>,
   (595, 5.0)),
  0.0),
 ((<Compressed Sparse Row sparse matrix of dtype 'float64'
   	with 1 stored elements and shape (1, 184)>,
   (594, 5.0)),
  0.0),
 ((<Compressed Sparse Row sparse matrix of dtype 'float64'
   	with 1 stored elements and shape (1, 184)>,
   (616, 5.0)),
  0.0),
 ((<Compressed Sparse Row sparse matrix of dtype 'float64'
   	with 1 stored elements and shape (1, 184)>,
   (588, 5.0)),
  0.0)]

In [34]:
# compute recommendation
df_recommendation = df_features[df_features['movieId'].isin(candidate_ids)][['movieId', 'feature_vector']]
df_recommendation['feature_vector'].apply(
    lambda v: lsh_usr.query(v, num_results=5, distance_func='cosine')
)

def predict_rating(hits):
    num, den = 0.0, 0.0
    for (_, (_, rating)), dist in hits:
        sim = 1 - dist
        num += sim * rating
        den += sim
    return num/den if den > 0 else None

df_recommendation['predicted_rating'] = df_recommendation['feature_vector'].apply(
    lambda v: predict_rating(lsh_usr.query(v, num_results=5, distance_func='cosine'))
)

df_recommendation = df_recommendation.sort_values('predicted_rating', ascending=False)
df_recommendation

,movieId,feature_vector,predicted_rating
97,588,<Compressed Sparse Row sparse matrix of dtype ...,5.0
101,594,<Compressed Sparse Row sparse matrix of dtype ...,5.0
102,595,<Compressed Sparse Row sparse matrix of dtype ...,5.0
103,596,<Compressed Sparse Row sparse matrix of dtype ...,5.0
106,616,<Compressed Sparse Row sparse matrix of dtype ...,5.0
164,1022,<Compressed Sparse Row sparse matrix of dtype ...,5.0
165,1025,<Compressed Sparse Row sparse matrix of dtype ...,5.0
166,1028,<Compressed Sparse Row sparse matrix of dtype ...,5.0
167,1029,<Compressed Sparse Row sparse matrix of dtype ...,5.0
168,1030,<Compressed Sparse Row sparse matrix of dtype ...,5.0


## Collaborative filtering

In [35]:
from surprise import SVD
from surprise import Reader
from surprise import Dataset
from surprise.model_selection import cross_validate
from surprise.prediction_algorithms.knns import KNNBasic

In [36]:
directory = 'data/movielens/ml-latest-small'
#directory = 'data/movielens/ml-latest' #change into this one for the full dataset (slow)
df_ratings = pd.read_csv(f'{directory}/ratings.csv')
df_ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [38]:
# initialize a data reader
reader = Reader(rating_scale=(1, 5))
# provide a dataset with userid, itemtid, and rating in order
data = Dataset.load_from_df(df_ratings[['userId','movieId','rating']], reader)

# surprise has also some built-in datasets that can be imported directly
#data = Dataset.load_builtin('ml-100k')

In [39]:
# initialize a user-based K nearest neighbors implementation
algo = KNNBasic(sim_options={'name': 'cosine', 'user_based': True}) # cosine/pearson/msd
# execute 5-fold cross-validation and measure RMSE and MAE
cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNBasic on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9810  0.9752  0.9744  0.9623  0.9728  0.9731  0.0061  
MAE (testset)     0.7568  0.7516  0.7490  0.7415  0.7472  0.7492  0.0050  
Fit time          0.20    0.24    0.23    0.23    0.24    0.23    0.01    
Test time         1.47    1.54    1.67    1.46    1.47    1.52    0.08    


{'test_rmse': array([0.98100472, 0.97517298, 0.97441945, 0.96229892, 0.97278267]),
 'test_mae': array([0.75682009, 0.75163825, 0.74902524, 0.74152015, 0.74721427]),
 'fit_time': (0.20090937614440918,
  0.23821520805358887,
  0.22898483276367188,
  0.23074817657470703,
  0.23519039154052734),
 'test_time': (1.4673192501068115,
  1.5434954166412354,
  1.6675550937652588,
  1.4572458267211914,
  1.4667506217956543)}